# Replication Notebook
## Forecasting Short-Term Semiconductor ETF Volatility: A Multi-Layer HAR Model with Microstructure Indicators and NLP Sentiment

**Author:** Alen Sahinpasic  
**Email:** sahinpasic_alen@student.ceu.edu  
**Date:** May 2026

---

### Purpose

This notebook is the single canonical replication pipeline for the thesis. It reconstructs the full empirical analysis from two inputs only:

1. **`data/LSEG_headlines_clean.csv`** — the cleaned LSEG Eikon semiconductor headline corpus (provided).
2. **Public OHLCV market data** — downloaded live from Yahoo Finance via `yfinance`.

Everything else — Garman-Klass volatility, forecast targets, microstructure variables, FinBERT sentiment scores, HAR model forecasts, loss metrics, DM tests, all tables, and all figures — is regenerated from scratch inside this notebook.

### Models Estimated

| Model | Specification |
|-------|---------------|
| **M1** | HAR-GK baseline: `const + gk_d + gk_w + gk_m` |
| **M2** | HAR + FinBERT sentiment: `M1 + S_t` |
| **M3** | HAR + microstructure: `M1 + range + relvol` |

All models use expanding-window OLS with Newey-West HAC standard errors (lag = h).  
In-sample: 2020-01-02 – 2024-04-30 | Out-of-sample: 2024-05-01 – 2025-12-31  
ETFs: SMH, SOXX | Horizons: h = 1, 2, 3, 4, 5

### Expected Runtime

| Section | Estimated time |
|---------|----------------|
| Market data download (yfinance) | < 1 min |
| GK / targets / microstructure | < 1 min |
| FinBERT scoring (~23,000 headlines) | **20–40 min CPU / 5–10 min MPS / 3–5 min CUDA** |
| HAR expanding-window OLS (3 models × 2 ETFs × 5 horizons) | 5–15 min |
| Tables and figures | < 2 min |

### Hardware Notes

- **Apple Silicon (MPS):** FinBERT scoring uses Apple MPS automatically if available. Recommended for Mac users.
- **CUDA:** If a CUDA GPU is detected, it is used preferentially over MPS.
- **CPU fallback:** All sections work on CPU; FinBERT scoring is slower but produces identical results.
- The notebook does **not** require a GPU for any section other than FinBERT scoring.

### Reproducibility Note

`LSEG_headlines_clean.csv` is the cleaned starting point and is the only provided input. Market data is re-downloaded each run; minor differences in OHLCV data beyond the original sample end date (2025-12-31) will not affect thesis results, which are computed on a fixed date range.

---
## Section 2 — Imports and Project Paths

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
import json
import numpy as np
import pandas as pd
import scipy.stats as sp_stats
import statsmodels.api as sm
from scipy.stats import gaussian_kde
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

import yfinance as yf
import torch
from transformers import pipeline as hf_pipeline

print(f"Python  {sys.version.split()[0]}")
print(f"pandas  {pd.__version__}")
print(f"numpy   {np.__version__}")
print(f"torch   {torch.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"CUDA available: {torch.cuda.is_available()}")

Python  3.9.12
pandas  2.2.3
numpy   1.26.4
torch   2.2.2
MPS available: True
CUDA available: False


In [2]:
# ── Project paths ──────────────────────────────────────────────────────────────
ROOT    = Path(".").resolve()          # submission_1/
DATA    = ROOT / "data"
RESULTS = ROOT / "results"
TABLES  = RESULTS / "tables"
FIGURES = RESULTS / "figures"

for d in [DATA, RESULTS, TABLES, FIGURES, TABLES / "formatted"]:
    d.mkdir(parents=True, exist_ok=True)

# ── Global constants ───────────────────────────────────────────────────────────
ETFS     = ["SMH", "SOXX"]
HORIZONS = [1, 2, 3, 4, 5]
SPLIT    = pd.Timestamp("2024-05-01")
START    = "2020-01-01"
END      = "2026-01-01"
BATCH_SIZE = 32

# ── Plot style ─────────────────────────────────────────────────────────────────
STYLE = {
    "font.family":       "serif",
    "font.size":         10,
    "axes.titlesize":    11,
    "axes.labelsize":    10,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "legend.fontsize":   9,
    "figure.dpi":        150,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "grid.linestyle":    "--",
}
plt.rcParams.update(STYLE)

COLORS = {"SMH": "#2166ac", "SOXX": "#d6604d",
          "M1": "#636363",  "M2": "#1a9641", "M3": "#E15759"}
YEAR_COLORS = {2020:"#4E79A7", 2021:"#F28E2B", 2022:"#E15759",
               2023:"#76B7B2", 2024:"#59A14F", 2025:"#B07AA1"}

print(f"ROOT    = {ROOT}")
print(f"DATA    = {DATA}")
print(f"RESULTS = {RESULTS}")
print(f"TABLES  = {TABLES}")
print(f"FIGURES = {FIGURES}")

ROOT    = /Volumes/KINGSTON/Thesis/submission_1
DATA    = /Volumes/KINGSTON/Thesis/submission_1/data
RESULTS = /Volumes/KINGSTON/Thesis/submission_1/results
TABLES  = /Volumes/KINGSTON/Thesis/submission_1/results/tables
FIGURES = /Volumes/KINGSTON/Thesis/submission_1/results/figures


---
## Section 3 — Load Cleaned LSEG Headlines

In [3]:
LSEG_PATH = DATA / "LSEG_headlines_clean.csv"
assert LSEG_PATH.exists(), f"Input file not found: {LSEG_PATH}"

hl_full = pd.read_csv(LSEG_PATH, dtype=str)

# ── Validate required columns ──────────────────────────────────────────────────
required_cols = {"date", "time", "source", "ticker", "headline"}
missing = required_cols - set(hl_full.columns)
assert not missing, f"Missing columns: {missing}"
print(f"Required columns present: {sorted(required_cols)}")

# ── Parse dates ────────────────────────────────────────────────────────────────
hl_full["date"] = pd.to_datetime(hl_full["date"], errors="coerce")

# ── Diagnostics ────────────────────────────────────────────────────────────────
total_rows   = len(hl_full)
news_rows    = hl_full["headline"].notna().sum()
zero_rows    = hl_full["headline"].isna().sum()
valid_dates  = hl_full["date"].notna().sum()
date_min     = hl_full["date"].min()
date_max     = hl_full["date"].max()
n_dup        = hl_full[hl_full["headline"].notna()].duplicated(subset=["date","headline"]).sum()

print(f"\nTotal rows          : {total_rows:,}")
print(f"Headline rows       : {news_rows:,}")
print(f"Zero-news rows      : {zero_rows:,}")
print(f"Valid dates         : {valid_dates:,}")
print(f"Date range          : {date_min.date()} → {date_max.date()}")
print(f"Duplicate headlines : {n_dup:,}")

# Headlines for FinBERT (news days only)
hl_news = hl_full[hl_full["headline"].notna()].copy().reset_index(drop=True)
print(f"\nHeadlines to score  : {len(hl_news):,}")

Required columns present: ['date', 'headline', 'source', 'ticker', 'time']

Total rows          : 23,472
Headline rows       : 23,430
Zero-news rows      : 42
Valid dates         : 23,472
Date range          : 2020-01-01 → 2025-12-31
Duplicate headlines : 0

Headlines to score  : 23,430


---
## Section 4 — Download Market Data (yfinance)

In [4]:
print(f"Downloading OHLCV for {ETFS}, {START} – {END} ...")

raw = yf.download(
    ETFS,
    start=START,
    end=END,
    auto_adjust=True,
    progress=True,
)

print(f"Downloaded shape: {raw.shape}")
print(f"Date range: {raw.index[0].date()} → {raw.index[-1].date()}")

# ── Save per-ticker OHLCV ──────────────────────────────────────────────────────
ohlcv = {}
for ticker in ETFS:
    cols = ["Open", "High", "Low", "Close", "Volume"]
    df_t = pd.DataFrame({c: raw[c][ticker] for c in cols}).dropna()
    out_path = DATA / f"ohlcv_{ticker}.csv"
    df_t.to_csv(out_path)
    ohlcv[ticker] = df_t
    print(f"  {ticker}: {len(df_t)} trading days → {out_path.name}")

# ── Validation ─────────────────────────────────────────────────────────────────
for ticker, df_t in ohlcv.items():
    assert df_t[["Open","High","Low","Close"]].notna().all().all(), f"{ticker}: NaN in OHLC"
    assert (df_t["High"] >= df_t["Low"]).all(), f"{ticker}: High < Low on some days"
    assert (df_t[["Open","High","Low","Close"]] > 0).all().all(), f"{ticker}: non-positive prices"
    assert (df_t["Volume"] >= 0).all(), f"{ticker}: negative volume"
    print(f"  [{ticker}] validation passed — {len(df_t)} rows, no OHLC issues")

[*********************100%***********************]  2 of 2 completed

Downloaded shape: (1508, 10)
Date range: 2020-01-02 → 2025-12-31
  SMH: 1508 trading days → ohlcv_SMH.csv
  SOXX: 1508 trading days → ohlcv_SOXX.csv
  [SMH] validation passed — 1508 rows, no OHLC issues
  [SOXX] validation passed — 1508 rows, no OHLC issues


---
## Section 5 — Compute Garman-Klass Volatility

In [5]:
def compute_gk(open_, high, low, close):
    """GK_t = sqrt(0.5*[ln(H/L)]^2 - (2*ln2-1)*[ln(C/O)]^2), floored at 0."""
    log_hl = np.log(high / low)
    log_co = np.log(close / open_)
    gk_sq  = 0.5 * log_hl**2 - (2 * np.log(2) - 1) * log_co**2
    return np.sqrt(np.maximum(gk_sq, 0.0))

gk_records = {}
for ticker in ETFS:
    df_t = ohlcv[ticker]
    gk_s = compute_gk(
        df_t["Open"].values,
        df_t["High"].values,
        df_t["Low"].values,
        df_t["Close"].values,
    )
    gk_records[ticker] = pd.Series(gk_s, index=df_t.index)

gk_df = pd.DataFrame(gk_records).dropna()
gk_df.index.name = "Date"

# ── Validation ─────────────────────────────────────────────────────────────────
assert (gk_df >= 0).all().all(), "GK contains negative values"
assert np.isfinite(gk_df.values).all(), "GK contains non-finite values"

for ticker in ETFS:
    s = gk_df[ticker]
    print(f"{ticker}: N={len(s)}  mean={s.mean():.6f}  std={s.std():.6f}  "
          f"min={s.min():.6f}  max={s.max():.6f}")

gk_path = DATA / "gk_volatility.csv"
gk_df.to_csv(gk_path)
print(f"\nSaved: {gk_path.name}  ({len(gk_df)} rows)")

SMH: N=1508  mean=0.014570  std=0.007768  min=0.002897  max=0.087393
SOXX: N=1508  mean=0.014940  std=0.008108  min=0.003432  max=0.091161

Saved: gk_volatility.csv  (1508 rows)


---
## Section 6 — Compute Forecast Targets (h = 1 … 5)

In [6]:
# RV_{t,h} = (1/h) * sum_{j=1}^h GK_{t+j}
# Implemented as: shift(-1) then rolling(h).mean() then shift(-(h-1))

targets = {}
for ticker in ETFS:
    gk_s = gk_df[ticker]
    for h in HORIZONS:
        gk_shifted = gk_s.shift(-1)
        target = gk_shifted.rolling(h, min_periods=h).mean().shift(-(h - 1))
        targets[f"{ticker}_h{h}"] = target

targets_df = pd.DataFrame(targets, index=gk_df.index)
targets_df.index.name = "Date"

# ── Validation ─────────────────────────────────────────────────────────────────
# NaNs should only appear at the tail (last h-1 rows)
for ticker in ETFS:
    for h in HORIZONS:
        col  = f"{ticker}_h{h}"
        n_na = targets_df[col].isna().sum()
        # NaNs only at tail
        first_nan = targets_df[col].isna().idxmax() if n_na > 0 else None
        if first_nan is not None:
            tail_only = targets_df[col].loc[first_nan:].isna().all()
            assert tail_only, f"{col}: NaNs not confined to tail"
        print(f"  {col}: {targets_df[col].notna().sum()} valid, {n_na} NaN (tail)")

tgt_path = DATA / "forecast_targets.csv"
targets_df.to_csv(tgt_path)
print(f"\nSaved: {tgt_path.name}")

  SMH_h1: 1507 valid, 1 NaN (tail)
  SMH_h2: 1506 valid, 2 NaN (tail)
  SMH_h3: 1505 valid, 3 NaN (tail)
  SMH_h4: 1504 valid, 4 NaN (tail)
  SMH_h5: 1503 valid, 5 NaN (tail)
  SOXX_h1: 1507 valid, 1 NaN (tail)
  SOXX_h2: 1506 valid, 2 NaN (tail)
  SOXX_h3: 1505 valid, 3 NaN (tail)
  SOXX_h4: 1504 valid, 4 NaN (tail)
  SOXX_h5: 1503 valid, 5 NaN (tail)

Saved: forecast_targets.csv


---
## Section 7 — Compute Microstructure Variables

In [7]:
micro = {}
for ticker in ETFS:
    df_t = ohlcv[ticker]
    # log high-low range
    range_var = np.log(df_t["High"] / df_t["Low"])
    # relative volume: volume / 20-day rolling mean
    rel_vol   = df_t["Volume"] / df_t["Volume"].rolling(20, min_periods=20).mean()
    micro[f"{ticker}_range"]  = range_var
    micro[f"{ticker}_relvol"] = rel_vol

micro_df = pd.DataFrame(micro)
micro_df.index.name = "Date"

# ── Validation ─────────────────────────────────────────────────────────────────
for ticker in ETFS:
    rng  = micro_df[f"{ticker}_range"].dropna()
    rvol = micro_df[f"{ticker}_relvol"].dropna()
    assert (rng > 0).all(), f"{ticker}_range: non-positive values"
    print(f"{ticker}_range : N={len(rng)}  mean={rng.mean():.4f}  min={rng.min():.4f}")
    print(f"{ticker}_relvol: N={len(rvol)}  mean={rvol.mean():.4f}  "
          f"(first 19 NaN from rolling window)")

micro_path = DATA / "microstructure.csv"
micro_df.to_csv(micro_path)
print(f"\nSaved: {micro_path.name}")

SMH_range : N=1508  mean=0.0245  min=0.0058
SMH_relvol: N=1489  mean=1.0052  (first 19 NaN from rolling window)
SOXX_range : N=1508  mean=0.0251  min=0.0054
SOXX_relvol: N=1489  mean=1.0189  (first 19 NaN from rolling window)

Saved: microstructure.csv


---
## Section 8 — FinBERT Headline Scoring

Scores every headline in `LSEG_headlines_clean.csv` using `ProsusAI/finbert`.  
Processes in batches of 32. Uses CUDA > MPS > CPU.  
On failure: retries once per batch; raises on second failure.

In [8]:
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

# ── Device selection ───────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = 0
    device_label = "CUDA"
elif torch.backends.mps.is_available():
    device = "mps"
    device_label = "Apple MPS"
else:
    device = -1
    device_label = "CPU"
print(f"FinBERT device: {device_label}")

# ── Load model ─────────────────────────────────────────────────────────────────
print("Loading ProsusAI/finbert ...")
pipe = hf_pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert",
    max_length=512,
    truncation=True,
    top_k=3,
    device=device,
)
print("FinBERT ready.")

# ── Score headlines ────────────────────────────────────────────────────────────
headlines_list = hl_news["headline"].tolist()
total          = len(headlines_list)
pos_scores, neg_scores, neu_scores = [], [], []

print(f"Scoring {total:,} headlines in batches of {BATCH_SIZE} ...")

for start in range(0, total, BATCH_SIZE):
    batch = headlines_list[start : start + BATCH_SIZE]

    # Retry once on failure
    for attempt in range(2):
        try:
            results = pipe(batch)
            break
        except Exception as e:
            if attempt == 0:
                print(f"  Batch {start//BATCH_SIZE} failed ({e}), retrying ...")
            else:
                raise RuntimeError(
                    f"FinBERT batch {start//BATCH_SIZE} failed twice at headline {start}: {e}"
                )

    for res in results:
        s = {d["label"]: d["score"] for d in res}
        pos_scores.append(s.get("positive", 0.0))
        neg_scores.append(s.get("negative", 0.0))
        neu_scores.append(s.get("neutral",  0.0))

    done = min(start + BATCH_SIZE, total)
    if done % 2000 < BATCH_SIZE or done == total:
        print(f"  Scored {done:,} / {total:,}")

# ── Assemble scored DataFrame ──────────────────────────────────────────────────
scored_df = pd.DataFrame({
    "date":        hl_news["date"].values,
    "time":        hl_news["time"].values,
    "source":      hl_news["source"].values,
    "ticker":      hl_news["ticker"].values,
    "headline":    hl_news["headline"].values,
    "positive":    pos_scores,
    "negative":    neg_scores,
    "neutral":     neu_scores,
    "net_sentiment": np.array(pos_scores) - np.array(neg_scores),
})
scored_df["date"] = pd.to_datetime(scored_df["date"])

scored_path = DATA / "headlines_scored.csv"
scored_df.to_csv(scored_path, index=False)
print(f"\nSaved: {scored_path.name}  ({len(scored_df):,} rows)")
print(f"Net sentiment — mean={scored_df['net_sentiment'].mean():.4f}  "
      f"std={scored_df['net_sentiment'].std():.4f}")

FinBERT device: Apple MPS
Loading ProsusAI/finbert ...
FinBERT ready.
Scoring 23,430 headlines in batches of 32 ...
  Scored 2,016 / 23,430
  Scored 4,000 / 23,430
  Scored 6,016 / 23,430
  Scored 8,000 / 23,430
  Scored 10,016 / 23,430
  Scored 12,000 / 23,430
  Scored 14,016 / 23,430
  Scored 16,000 / 23,430
  Scored 18,016 / 23,430
  Scored 20,000 / 23,430
  Scored 22,016 / 23,430
  Scored 23,430 / 23,430

Saved: headlines_scored.csv  (23,430 rows)
Net sentiment — mean=-0.0297  std=0.6082


---
## Section 9 — Daily Sentiment Aggregation

In [9]:
# Trading calendar = GK volatility dates
trading_days = gk_df.index

# Aggregate per-headline scores to daily
# S_t = mean(positive - negative) across all headlines on date t
daily_agg = (
    scored_df.groupby("date")
    .agg(
        S_t         = ("net_sentiment", "mean"),
        S_pos_t     = ("positive",      "mean"),
        S_neg_t     = ("negative",      "mean"),
        n_headlines = ("headline",      "count"),
    )
    .reset_index()
)
daily_agg["date"] = pd.to_datetime(daily_agg["date"])

# Align to trading calendar; zero-fill missing days
cal = pd.DataFrame({"date": trading_days}).merge(daily_agg, on="date", how="left")
cal["n_headlines"] = cal["n_headlines"].fillna(0).astype(int)
cal["S_t"]         = cal["S_t"].fillna(0.0)
cal["S_pos_t"]     = cal["S_pos_t"].fillna(0.0)
cal["S_neg_t"]     = cal["S_neg_t"].fillna(0.0)
cal = cal.sort_values("date").reset_index(drop=True)

sent_path = DATA / "daily_sentiment.csv"
cal.to_csv(sent_path, index=False)

# ── Validation and diagnostics ─────────────────────────────────────────────────
news_days      = (cal["n_headlines"] > 0).sum()
zero_news_days = (cal["n_headlines"] == 0).sum()

print(f"Trading days total : {len(cal):,}")
print(f"Days with news     : {news_days:,}")
print(f"Zero-news days     : {zero_news_days:,}")
print(f"S_t  mean={cal['S_t'].mean():.4f}  std={cal['S_t'].std():.4f}  "
      f"min={cal['S_t'].min():.4f}  max={cal['S_t'].max():.4f}")
print(f"Calendar start     : {cal['date'].min().date()}")
print(f"Calendar end       : {cal['date'].max().date()}")
print(f"\nSaved: {sent_path.name}  ({len(cal)} rows)")

Trading days total : 1,508
Days with news     : 1,466
Zero-news days     : 42
S_t  mean=-0.0307  std=0.2680  min=-0.9638  max=0.9394
Calendar start     : 2020-01-02
Calendar end       : 2025-12-31

Saved: daily_sentiment.csv  (1508 rows)


---
## Section 10 — Prepare HAR Modelling Dataset

In [10]:
# ── Load all series ────────────────────────────────────────────────────────────
gk    = pd.read_csv(DATA / "gk_volatility.csv",    index_col=0, parse_dates=True)
tgt   = pd.read_csv(DATA / "forecast_targets.csv", index_col=0, parse_dates=True)
micro = pd.read_csv(DATA / "microstructure.csv",   index_col=0, parse_dates=True)
sent  = pd.read_csv(DATA / "daily_sentiment.csv")
sent["date"] = pd.to_datetime(sent["date"])
sent  = sent.set_index("date")[["S_t"]]

# Align to common trading calendar
common = gk.index.intersection(tgt.index).intersection(micro.index)
gk     = gk.loc[common]
tgt    = tgt.loc[common]
micro  = micro.loc[common]
sent   = sent.reindex(common).fillna(0.0)

# ── Sample split ───────────────────────────────────────────────────────────────
is_idx  = common[common <  SPLIT]
oos_idx = common[common >= SPLIT]

print(f"Common trading days : {len(common)}")
print(f"In-sample (IS)      : {len(is_idx)}  [{is_idx[0].date()} – {is_idx[-1].date()}]")
print(f"Out-of-sample (OOS) : {len(oos_idx)} [{oos_idx[0].date()} – {oos_idx[-1].date()}]")

# ── HAR components ─────────────────────────────────────────────────────────────
# gk_d = lag-1 GK
# gk_w = 5-day rolling mean of lag-1 GK
# gk_m = 22-day rolling mean of lag-1 GK
har = {}
for etf in ETFS:
    g = gk[etf]
    har[etf] = pd.DataFrame({
        "gk_d": g.shift(1),
        "gk_w": g.shift(1).rolling(5,  min_periods=5).mean(),
        "gk_m": g.shift(1).rolling(22, min_periods=22).mean(),
    })
    print(f"{etf} HAR components built: "
          f"gk_d={har[etf]['gk_d'].notna().sum()} "
          f"gk_w={har[etf]['gk_w'].notna().sum()} "
          f"gk_m={har[etf]['gk_m'].notna().sum()} valid obs")

Common trading days : 1508
In-sample (IS)      : 1089  [2020-01-02 – 2024-04-30]
Out-of-sample (OOS) : 419 [2024-05-01 – 2025-12-31]
SMH HAR components built: gk_d=1507 gk_w=1503 gk_m=1486 valid obs
SOXX HAR components built: gk_d=1507 gk_w=1503 gk_m=1486 valid obs


---
## Section 11 — Model Estimation Functions

In [11]:
def qlike_loss(realized, forecast):
    """QLIKE loss: mean(log(f) + r/f). f floored at 1e-10."""
    f = np.maximum(forecast, 1e-10)
    return float(np.mean(np.log(f) + realized / f))


def newey_west_dm(d, h):
    """Newey-West HAC DM test. d = loss_baseline - loss_model (positive = model better)."""
    d = d[~np.isnan(d)]
    n = len(d)
    if n < 2:
        return np.nan, np.nan
    d_bar  = d.mean()
    gamma0 = np.mean((d - d_bar) ** 2)
    gammas = 0.0
    for k in range(1, h + 1):
        w = 1 - k / (h + 1)
        gammas += 2 * w * np.mean((d[k:] - d_bar) * (d[:-k] - d_bar))
    lrv = gamma0 + gammas
    if lrv <= 0:
        return np.nan, np.nan
    dm  = d_bar / np.sqrt(lrv / n)
    p   = 2 * (1 - sp_stats.norm.cdf(abs(dm)))
    return float(dm), float(p)


def expanding_window_ols(X_full, y_full, is_idx, oos_idx, h):
    """
    Expanding-window OLS forecast.
    For each OOS date, trains on IS+preceding OOS up to (not including) that date.
    Returns forecasts Series and coefficient snapshot list.
    """
    forecasts    = pd.Series(np.nan, index=oos_idx)
    coef_records = []
    is_start     = X_full.index.get_loc(is_idx[0])

    for i, date in enumerate(oos_idx):
        end_loc = X_full.index.get_loc(date)
        X_tr    = X_full.iloc[is_start:end_loc]
        y_tr    = y_full.iloc[is_start:end_loc]

        valid   = X_tr.notna().all(axis=1) & y_tr.notna()
        X_tr, y_tr = X_tr.loc[valid], y_tr.loc[valid]

        if len(y_tr) < len(X_tr.columns) + 5:
            continue

        try:
            res   = sm.OLS(y_tr, X_tr).fit(
                cov_type="HAC", cov_kwds={"maxlags": max(h, 1)}, use_t=False
            )
            x_new = X_full.loc[date]
            if x_new.notna().all():
                forecasts.loc[date] = float(res.params @ x_new)

            if i == 0 or i == len(oos_idx) - 1 or i % 100 == 0:
                for var in X_tr.columns:
                    coef_records.append({
                        "step": i, "variable": var,
                        "coefficient": res.params[var],
                        "std_error":   res.bse[var],
                        "t_stat":      res.tvalues[var],
                        "p_value":     res.pvalues[var],
                    })
        except Exception as e:
            raise RuntimeError(f"OLS failed at date={date}, h={h}: {e}")

    return forecasts, coef_records


def compute_loss_metrics(realized, forecast, is_mean_gk):
    """
    Replace non-positive forecasts with in-sample mean GK.
    Returns dict of RMSE, MAE, QLIKE, N_forecasts, N_replaced.
    """
    f = forecast.copy()
    bad = f.notna() & (f<= 0)
    f[bad] = is_mean_gk
    mask = f.notna() & realized.notna()
    f_v, r_v = f.loc[mask].values, realized.loc[mask].values
    return {
        "RMSE":        float(np.sqrt(np.mean((r_v - f_v) ** 2))),
        "MAE":         float(np.mean(np.abs(r_v - f_v))),
        "QLIKE":       qlike_loss(r_v, f_v),
        "N_forecasts": int(mask.sum()),
        "N_replaced":  int(bad.sum()),
    }, f


def star(p):
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.10: return "*"
    return ""


print("Estimation functions defined: expanding_window_ols, compute_loss_metrics, newey_west_dm")

Estimation functions defined: expanding_window_ols, compute_loss_metrics, newey_west_dm


---
## Section 12 — Estimate M1 / M2 / M3 (Expanding-Window OLS)

In [12]:
# Storage
forecasts  = {etf: {} for etf in ETFS}   # forecasts[etf]["M1_h1"] = Series
coef_rows  = []                           # coefficient snapshots
is_results = {}                           # in-sample OLS results for table/fig

for etf in ETFS:
    print(f"\n{'='*50}\n  {etf}\n{'='*50}")
    micro_sub = micro[[f"{etf}_range", f"{etf}_relvol"]].rename(
        columns={f"{etf}_range": "range", f"{etf}_relvol": "relvol"})

    for h in HORIZONS:
        y = tgt[f"{etf}_h{h}"]

        # ── M1: HAR baseline ──
        X1 = sm.add_constant(har[etf][["gk_d","gk_w","gk_m"]])
        X1.columns = ["const","gk_d","gk_w","gk_m"]
        f1, c1 = expanding_window_ols(X1, y, is_idx, oos_idx, h)
        forecasts[etf][f"M1_h{h}"] = f1
        for r in c1:
            r.update({"ETF": etf, "horizon": h, "model": "M1"})
        coef_rows.extend(c1)

        # ── M2: HAR + FinBERT sentiment ──
        X2 = pd.concat([
            sm.add_constant(har[etf][["gk_d","gk_w","gk_m"]]),
            sent.rename(columns={"S_t": "sentiment"}),
        ], axis=1)
        X2.columns = ["const","gk_d","gk_w","gk_m","sentiment"]
        f2, c2 = expanding_window_ols(X2, y, is_idx, oos_idx, h)
        forecasts[etf][f"M2_h{h}"] = f2
        for r in c2:
            r.update({"ETF": etf, "horizon": h, "model": "M2"})
        coef_rows.extend(c2)

        # ── M3: HAR + microstructure ──
        X3 = pd.concat([X1, micro_sub], axis=1)
        f3, c3 = expanding_window_ols(X3, y, is_idx, oos_idx, h)
        forecasts[etf][f"M3_h{h}"] = f3
        for r in c3:
            r.update({"ETF": etf, "horizon": h, "model": "M3"})
        coef_rows.extend(c3)

        # ── In-sample OLS (for tables and Fig 4) ──
        for model_name, X_is in [("M1", X1), ("M2", X2), ("M3", X3)]:
            y_is   = y.loc[is_idx]
            X_is_s = X_is.loc[is_idx]
            mask   = X_is_s.notna().all(axis=1) & y_is.notna()
            res    = sm.OLS(y_is[mask], X_is_s[mask]).fit(
                cov_type="HAC", cov_kwds={"maxlags": max(h, 1)}, use_t=False)
            is_results[(etf, h, model_name)] = res

        print(f"  h={h}: M1 n={f1.notna().sum()}  M2 n={f2.notna().sum()}  M3 n={f3.notna().sum()}")

# ── Save coefficient snapshots ─────────────────────────────────────────────────
coef_df = pd.DataFrame(coef_rows)
coef_cols = ["ETF","horizon","model","variable","coefficient","std_error","t_stat","p_value"]
coef_df[coef_cols].to_csv(RESULTS / "coefficients.csv", index=False)
print(f"\nSaved: coefficients.csv  ({len(coef_df)} rows)")


  SMH


  h=1: M1 n=419  M2 n=419  M3 n=419
  h=2: M1 n=419  M2 n=419  M3 n=419
  h=3: M1 n=419  M2 n=419  M3 n=419
  h=4: M1 n=419  M2 n=419  M3 n=419
  h=5: M1 n=419  M2 n=419  M3 n=419

  SOXX
  h=1: M1 n=419  M2 n=419  M3 n=419
  h=2: M1 n=419  M2 n=419  M3 n=419
  h=3: M1 n=419  M2 n=419  M3 n=419
  h=4: M1 n=419  M2 n=419  M3 n=419
  h=5: M1 n=419  M2 n=419  M3 n=419

Saved: coefficients.csv  (900 rows)


---
## Section 13 — Loss Metrics (RMSE, MAE, QLIKE)

In [13]:
loss_rows = []
fc_dfs    = {}   # fc_dfs[etf] = DataFrame with realized + forecasts

for etf in ETFS:
    is_mean = float(gk.loc[is_idx, etf].mean())
    rows_etf = {"date": oos_idx}

    for h in HORIZONS:
        realized = tgt.loc[oos_idx, f"{etf}_h{h}"]
        rows_etf[f"realized_h{h}"] = realized.values

        for model in ["M1", "M2", "M3"]:
            raw_fc = forecasts[etf][f"{model}_h{h}"].copy()
            metrics, fc_clean = compute_loss_metrics(realized, raw_fc, is_mean)
            rows_etf[f"{model}_h{h}_forecast"] = fc_clean.values
            loss_rows.append({"ETF": etf, "horizon": h, "model": model, **metrics})

    fc_dfs[etf] = pd.DataFrame(rows_etf)

loss_df = pd.DataFrame(loss_rows)

# Save forecast files
for etf in ETFS:
    fc_dfs[etf].to_csv(RESULTS / f"forecasts_{etf}.csv", index=False)
    print(f"Saved forecasts_{etf}.csv  ({len(fc_dfs[etf])} rows)")

# Save loss metrics
loss_df.to_csv(RESULTS / "loss_metrics.csv", index=False)
print(f"Saved loss_metrics.csv  ({len(loss_df)} rows)")

# Display summary
print("\nRMSE summary (h=1):")
h1_loss = loss_df[loss_df["horizon"] == 1].pivot_table(
    index="ETF", columns="model", values="RMSE")
print(h1_loss.round(6).to_string())

Saved forecasts_SMH.csv  (419 rows)
Saved forecasts_SOXX.csv  (419 rows)
Saved loss_metrics.csv  (30 rows)

RMSE summary (h=1):
model        M1        M2        M3
ETF                                
SMH    0.007474  0.007426  0.006845
SOXX   0.007691  0.007652  0.007051


---
## Section 14 — Diebold-Mariano Tests (Bonferroni-corrected)

In [14]:
N_H = len(HORIZONS)
dm_rows = []

for etf in ETFS:
    for h in HORIZONS:
        realized = tgt.loc[oos_idx, f"{etf}_h{h}"]

        def sq_err(model):
            f = fc_dfs[etf].set_index("date")[f"{model}_h{h}_forecast"]
            f.index = pd.to_datetime(f.index)
            return (realized - f) ** 2

        se1, se2, se3 = sq_err("M1"), sq_err("M2"), sq_err("M3")

        for comp, se_base, se_model, model_label in [
            ("M2_vs_M1", se1, se2, "M2 vs M1"),
            ("M3_vs_M1", se1, se3, "M3 vs M1"),
            ("M2_vs_M3", se3, se2, "M2 vs M3"),
        ]:
            mask = se_base.notna() & se_model.notna() & realized.notna()
            d = (se_base - se_model).loc[mask].values

            dm_s, p_raw = newey_west_dm(d, h)
            p_bonf = min(p_raw * N_H, 1.0) if not np.isnan(p_raw) else np.nan

            dm_rows.append({
                "ETF": etf,
                "horizon": h,
                "comparison": comp,
                "DM_stat": dm_s,
                "p_value_raw": p_raw,
                "p_value_bonferroni": p_bonf,
                "significant_5pct": not np.isnan(p_bonf) and p_bonf < 0.05,
                "N": int(mask.sum()),
            })

            sig = "*" if not np.isnan(p_bonf) and p_bonf < 0.05 else ""
            print(
                f"  {etf} h={h} {model_label:10s}: "
                f"DM={dm_s:6.3f}  p={p_raw:.4f}  p_bonf={p_bonf:.4f} {sig}"
            )

dm_df = pd.DataFrame(dm_rows)
dm_df.to_csv(RESULTS / "dm_tests.csv", index=False)
print(f"\nSaved: dm_tests.csv  ({len(dm_df)} rows)")

  SMH h=1 M2 vs M1  : DM= 2.801  p=0.0051  p_bonf=0.0254 *
  SMH h=1 M3 vs M1  : DM= 1.983  p=0.0474  p_bonf=0.2371 
  SMH h=1 M2 vs M3  : DM=-1.869  p=0.0616  p_bonf=0.3078 
  SMH h=2 M2 vs M1  : DM= 3.001  p=0.0027  p_bonf=0.0135 *
  SMH h=2 M3 vs M1  : DM= 1.746  p=0.0807  p_bonf=0.4037 
  SMH h=2 M2 vs M3  : DM=-1.656  p=0.0977  p_bonf=0.4883 
  SMH h=3 M2 vs M1  : DM= 3.042  p=0.0024  p_bonf=0.0118 *
  SMH h=3 M3 vs M1  : DM= 1.807  p=0.0708  p_bonf=0.3538 
  SMH h=3 M2 vs M3  : DM=-1.688  p=0.0915  p_bonf=0.4574 
  SMH h=4 M2 vs M1  : DM= 2.925  p=0.0034  p_bonf=0.0172 *
  SMH h=4 M3 vs M1  : DM= 1.796  p=0.0726  p_bonf=0.3628 
  SMH h=4 M2 vs M3  : DM=-1.607  p=0.1080  p_bonf=0.5399 
  SMH h=5 M2 vs M1  : DM= 3.011  p=0.0026  p_bonf=0.0130 *
  SMH h=5 M3 vs M1  : DM= 2.119  p=0.0341  p_bonf=0.1705 
  SMH h=5 M2 vs M3  : DM=-1.920  p=0.0548  p_bonf=0.2740 
  SOXX h=1 M2 vs M1  : DM= 2.401  p=0.0163  p_bonf=0.0817 
  SOXX h=1 M3 vs M1  : DM= 1.758  p=0.0788  p_bonf=0.3938 
  SOXX 

---
## Section 14B — Conditional Microstructure Gains

In [15]:
# Diagnose when the microstructure-augmented model adds the most value.
#
# Main comparison:
# M1 = HAR benchmark
# M3 = HAR + microstructure indicators
#
# Conditions are defined over the h=1 out-of-sample period.
# These are ex-post diagnostic groups, not real-time trading rules.

# Align calendar variables with OOS period
cal_oos = cal.set_index("date").reindex(oos_idx)
oos_headlines = cal_oos["n_headlines"]
oos_sent = cal_oos["S_t"]

# Threshold for large-news days
p75_headlines = oos_headlines.quantile(0.75)

cond_rows = []

for etf in ETFS:
    fc = fc_dfs[etf].copy()
    fc["date"] = pd.to_datetime(fc["date"])
    fc = fc.set_index("date")

    # One-day-ahead realized volatility and forecasts
    realized = fc["realized_h1"]
    f_M1 = fc["M1_h1_forecast"]   # HAR benchmark
    f_M3 = fc["M3_h1_forecast"]   # HAR + microstructure

    # Squared forecast errors
    se1 = (realized - f_M1) ** 2
    se3 = (realized - f_M3) ** 2

    # Align microstructure data with OOS period
    oos_range = micro.loc[oos_idx, f"{etf}_range"].reindex(realized.index)

    # Thresholds for diagnostic groups
    p75_vol = realized.quantile(0.75)
    p75_range = oos_range.quantile(0.75)

    conditions = {
        ("Realized volatility", "High-volatility days"):
            realized > p75_vol,

        ("Realized volatility", "Normal-volatility days"):
            realized <= p75_vol,

        ("News volume", "Large-news days"):
            oos_headlines.reindex(realized.index) > p75_headlines,

        ("News volume", "Quiet-news days"):
            oos_headlines.reindex(realized.index) <= p75_headlines,

        ("News sentiment", "Negative-sentiment days"):
            oos_sent.reindex(realized.index) < 0,

        ("News sentiment", "Non-negative-sentiment days"):
            oos_sent.reindex(realized.index) >= 0,

        ("Intraday price range", "High-range days"):
            oos_range > p75_range,

        ("Intraday price range", "Normal-range days"):
            oos_range <= p75_range,
    }

    for (condition, group), mask in conditions.items():
        mask = mask.reindex(realized.index).fillna(False)

        valid = (
            mask
            & realized.notna()
            & f_M1.notna()
            & f_M3.notna()
            & se1.notna()
            & se3.notna()
        )

        n = int(valid.sum())
        if n < 5:
            continue

        rmse_m1 = float(np.sqrt(se1[valid].mean()))
        rmse_m3 = float(np.sqrt(se3[valid].mean()))

        # Positive value means M3 has lower RMSE than M1
        rmse_improvement_pct = 100 * (rmse_m1 - rmse_m3) / rmse_m1

        # Mean squared-error improvement, retained in the raw results file
        mean_se_improvement = float((se1[valid] - se3[valid]).mean())

        cond_rows.append({
            "ETF": etf,
            "Condition": condition,
            "Group": group,
            "N": n,
            "RMSE_M1": rmse_m1,
            "RMSE_M3": rmse_m3,
            "M3_vs_M1_pct": rmse_improvement_pct,
            "mean_SE_improvement_vs_M1": mean_se_improvement,
        })

conditional_df = pd.DataFrame(cond_rows)

# Save full raw diagnostic results
conditional_df.to_csv(
    RESULTS / "conditional_microstructure_gains.csv",
    index=False
)

# Reader-facing table: no M2 columns, because the main comparison is M3 vs M1
tbl6_main = conditional_df[[
    "ETF",
    "Condition",
    "Group",
    "N",
    "RMSE_M1",
    "RMSE_M3",
    "M3_vs_M1_pct"
]].copy()

tbl6_main.columns = [
    "ETF",
    "Condition",
    "Group",
    "N",
    "HAR benchmark RMSE",
    "Microstructure RMSE",
    "RMSE improvement (%)"
]

# Clean rounding for thesis table
tbl6_main["HAR benchmark RMSE"] = tbl6_main["HAR benchmark RMSE"].round(4)
tbl6_main["Microstructure RMSE"] = tbl6_main["Microstructure RMSE"].round(4)
tbl6_main["RMSE improvement (%)"] = tbl6_main["RMSE improvement (%)"].round(1)

# Save thesis-ready table
tbl6_main.to_csv(
    TABLES / "table6_conditional_microstructure_gains_main.csv",
    index=False
)

print("Saved: conditional_microstructure_gains.csv")
print("Saved: table6_conditional_microstructure_gains_main.csv")
print()
print(tbl6_main.to_string(index=False))

print(
    "\nNote: Positive improvement means the microstructure model has lower RMSE "
    "\nthan the HAR benchmark. Conditions are defined over the h=1 out-of-sample "
    "\nperiod from May 2024 to December 2025. High-volatility and high-range days "
    "\nare days above the 75th percentile of realized volatility and log high-low "
    "\nrange, respectively. Large-news days are days above the 75th percentile of "
    "\ndaily headline counts. Negative-sentiment days are days with negative "
    "\naggregate FinBERT sentiment. These groups are ex-post diagnostic categories, "
    "not real-time trading rules."
)

Saved: conditional_microstructure_gains.csv
Saved: table6_conditional_microstructure_gains_main.csv

 ETF            Condition                       Group   N  HAR benchmark RMSE  Microstructure RMSE  RMSE improvement (%)
 SMH  Realized volatility        High-volatility days 105              0.0126               0.0113                   9.9
 SMH  Realized volatility      Normal-volatility days 313              0.0047               0.0044                   4.8
 SMH          News volume             Large-news days  95              0.0064               0.0060                   5.0
 SMH          News volume             Quiet-news days 323              0.0078               0.0071                   9.1
 SMH       News sentiment     Negative-sentiment days 205              0.0086               0.0076                  10.9
 SMH       News sentiment Non-negative-sentiment days 213              0.0063               0.0060                   4.1
 SMH Intraday price range             High-range day

In [33]:
tbl6_small = conditional_df[[
    "ETF",
    "Condition",
    "Group",
    "N",
    "M3_vs_M1_pct"
]].copy()

tbl6_small.columns = [
    "ETF",
    "Condition",
    "Group",
    "N",
    "RMSE improvement (%)"
]

tbl6_small["RMSE improvement (%)"] = tbl6_small["RMSE improvement (%)"].round(1)

tbl6_small.to_csv(
    TABLES / "table6_conditional_microstructure_gains_small.csv",
    index=False
)

print(tbl6_small.to_string(index=False))

 ETF            Condition                       Group   N  RMSE improvement (%)
 SMH  Realized volatility        High-volatility days 105                   9.9
 SMH  Realized volatility      Normal-volatility days 313                   4.8
 SMH          News volume             Large-news days  95                   5.0
 SMH          News volume             Quiet-news days 323                   9.1
 SMH       News sentiment     Negative-sentiment days 205                  10.9
 SMH       News sentiment Non-negative-sentiment days 213                   4.1
 SMH Intraday price range             High-range days 105                  11.7
 SMH Intraday price range           Normal-range days 313                   3.7
SOXX  Realized volatility        High-volatility days 105                   9.9
SOXX  Realized volatility      Normal-volatility days 313                   4.8
SOXX          News volume             Large-news days  95                   3.5
SOXX          News volume             Qu

In [32]:
# ── Figure G — Conditional RMSE Improvement of Microstructure Model ───────────

group_order = [
    "High-volatility days",
    "Normal-volatility days",
    "Large-news days",
    "Quiet-news days",
    "Negative-sentiment days",
    "Non-negative-sentiment days",
    "High-range days",
    "Normal-range days",
]

group_colors = {
    "High-volatility days": "#d73027",
    "Normal-volatility days": "#4575b4",
    "Large-news days": "#f46d43",
    "Quiet-news days": "#74add1",
    "Negative-sentiment days": "#a50026",
    "Non-negative-sentiment days": "#313695",
    "High-range days": "#e08214",
    "Normal-range days": "#8073ac",
}

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

fig.suptitle(
    "Figure G — RMSE Improvement from Adding Microstructure Indicators\n"
    "Relative to the HAR Benchmark, OOS Period May 2024 – December 2025",
    fontsize=11,
    y=1.02
)

for ax, etf in zip(axes, ["SMH", "SOXX"]):
    sub = conditional_df[conditional_df["ETF"] == etf].set_index("Group")

    vals = [
        sub.loc[g, "M3_vs_M1_pct"] if g in sub.index else 0
        for g in group_order
    ]

    ns = [
        int(sub.loc[g, "N"]) if g in sub.index else 0
        for g in group_order
    ]

    colors = [group_colors[g] for g in group_order]
    y_pos = np.arange(len(group_order))

    bars = ax.barh(
        y_pos,
        vals,
        color=colors,
        edgecolor="white",
        height=0.65
    )

    for bar, v, n in zip(bars, vals, ns):
        ax.text(
            v + (0.1 if v >= 0 else -0.1),
            bar.get_y() + bar.get_height() / 2,
            f"{v:+.1f}%  (n={n})",
            va="center",
            ha="left" if v >= 0 else "right",
            fontsize=8.5
        )

    ax.set_yticks(y_pos)
    ax.set_yticklabels(group_order, fontsize=9)

    # Put high-volatility/high-range groups at the top
    ax.invert_yaxis()

    ax.axvline(
        0,
        color="black",
        linewidth=0.8,
        linestyle="--"
    )

    ax.set_xlabel(
        "RMSE improvement (%)\n"
        "positive = microstructure model better than HAR benchmark",
        fontsize=9
    )

    ax.set_title(
        etf,
        fontsize=11,
        fontweight="bold"
    )

    ax.grid(
        axis="x",
        linestyle="--",
        alpha=0.35
    )

plt.tight_layout()

fig.savefig(
    FIGURES / "figG_conditional_microstructure_gains.png",
    dpi=150,
    bbox_inches="tight"
)

fig.savefig(
    FIGURES / "figG_conditional_microstructure_gains.pdf",
    bbox_inches="tight"
)

plt.close()

print("Saved: figG_conditional_microstructure_gains.png / .pdf")

Saved: figG_conditional_microstructure_gains.png / .pdf


---
## Section 15 — Final Table Generation

In [36]:
# ─────────────────────────────────────────────────────────────────────────────
# Thesis Tables — Revised Main Results Structure
# Main focus: M1 HAR benchmark vs M3 microstructure model
# M2 sentiment results are moved to appendix/secondary tables.
# ─────────────────────────────────────────────────────────────────────────────

# Helper function for clean table rounding
def round_for_thesis(df):
    df = df.copy()

    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            col_lower = col.lower()

            if col in ["N", "h", "Horizon"]:
                df[col] = df[col].astype(int)

            elif "%" in col or "pct" in col_lower or "improvement" in col_lower:
                df[col] = df[col].round(1)

            elif "p" in col_lower and "value" in col_lower:
                df[col] = df[col].round(3)

            elif "dm" in col_lower:
                df[col] = df[col].round(3)

            elif "qlike" in col_lower:
                df[col] = df[col].round(3)

            else:
                df[col] = df[col].round(4)

    return df


# ── Table 1 — Summary Statistics, Cleaned ────────────────────────────────────

stat_rows = []

for etf in ETFS:
    gk_s = gk.loc[is_idx, etf].dropna()

    s_s = (
        sent.loc[is_idx, "S_t"]
        if "S_t" in sent.columns
        else pd.Series(dtype=float)
    ).dropna()

    rng = micro.loc[is_idx, f"{etf}_range"].dropna()
    rvol = micro.loc[is_idx, f"{etf}_relvol"].dropna()

    for label, s in [
        ("Volatility", gk_s),
        ("News sentiment", s_s),
        ("Log high-low range", rng),
        ("Relative volume", rvol),
    ]:
        if len(s) == 0:
            continue

        stat_rows.append({
            "ETF": etf,
            "Variable": label,
            "N": int(s.notna().sum()),
            "Mean": s.mean(),
            "SD": s.std(),
            "Min": s.min(),
            "Median": s.median(),
            "Max": s.max(),
            "AC1": float(s.autocorr(1)),
            "AC5": float(s.autocorr(5)),
            "AC22": float(s.autocorr(22)),
        })

tbl1 = pd.DataFrame(stat_rows)
tbl1 = round_for_thesis(tbl1)

tbl1.to_csv(TABLES / "table1_summary_stats_clean.csv", index=False)
print("Saved: table1_summary_stats_clean.csv")


# ── Table 2 — Main Forecast Accuracy: M1 vs M3 ───────────────────────────────
# M1 = HAR benchmark
# M3 = HAR + microstructure indicators

tbl2_rows = []

for etf in ETFS:
    for h in HORIZONS:
        r1 = loss_df[
            (loss_df.ETF == etf) &
            (loss_df.horizon == h) &
            (loss_df.model == "M1")
        ].iloc[0]

        r3 = loss_df[
            (loss_df.ETF == etf) &
            (loss_df.horizon == h) &
            (loss_df.model == "M3")
        ].iloc[0]

        tbl2_rows.append({
            "ETF": etf,
            "Horizon": h,
            "HAR RMSE": r1.RMSE,
            "Microstructure RMSE": r3.RMSE,
            "RMSE improvement (%)": 100 * (r1.RMSE - r3.RMSE) / r1.RMSE,
            "HAR MAE": r1.MAE,
            "Microstructure MAE": r3.MAE,
            "MAE improvement (%)": 100 * (r1.MAE - r3.MAE) / r1.MAE,
            "N": int(r1.N_forecasts),
        })

tbl2 = pd.DataFrame(tbl2_rows)
tbl2 = round_for_thesis(tbl2)

tbl2.to_csv(TABLES / "table2_main_microstructure_vs_har.csv", index=False)
print("Saved: table2_main_microstructure_vs_har.csv")


# ── Appendix Table A1 — Sentiment Model vs HAR Benchmark ─────────────────────
# M1 = HAR benchmark
# M2 = HAR + FinBERT sentiment
# This is no longer the headline comparison.

a1_rows = []

for etf in ETFS:
    for h in HORIZONS:
        r1 = loss_df[
            (loss_df.ETF == etf) &
            (loss_df.horizon == h) &
            (loss_df.model == "M1")
        ].iloc[0]

        r2 = loss_df[
            (loss_df.ETF == etf) &
            (loss_df.horizon == h) &
            (loss_df.model == "M2")
        ].iloc[0]

        a1_rows.append({
            "ETF": etf,
            "Horizon": h,
            "HAR RMSE": r1.RMSE,
            "Sentiment RMSE": r2.RMSE,
            "RMSE improvement (%)": 100 * (r1.RMSE - r2.RMSE) / r1.RMSE,
            "HAR MAE": r1.MAE,
            "Sentiment MAE": r2.MAE,
            "MAE improvement (%)": 100 * (r1.MAE - r2.MAE) / r1.MAE,
            "HAR QLIKE": r1.QLIKE,
            "Sentiment QLIKE": r2.QLIKE,
            "QLIKE change": r2.QLIKE - r1.QLIKE,
            "N": int(r1.N_forecasts),
        })

app_a1 = pd.DataFrame(a1_rows)
app_a1 = round_for_thesis(app_a1)

app_a1.to_csv(TABLES / "appendix_table_A1_sentiment_vs_har.csv", index=False)
print("Saved: appendix_table_A1_sentiment_vs_har.csv")


# ── Table 3 — Main DM Tests: M3 vs M1 ────────────────────────────────────────
# Main statistical test for the headline comparison.

tbl3 = dm_df[dm_df["comparison"] == "M3_vs_M1"].copy()

# Rename columns if they exist in your dm_df
tbl3 = tbl3.rename(columns={
    "horizon": "Horizon",
    "DM_stat": "DM statistic",
    "dm_stat": "DM statistic",
    "p_value_raw": "Raw p-value",
    "p_raw": "Raw p-value",
    "p_value_bonferroni": "Bonferroni p-value",
    "p_bonf": "Bonferroni p-value",
    "significant_5pct": "Significant at 5%",
})

# Keep only readable columns that exist
wanted_cols = [
    "ETF",
    "Horizon",
    "DM statistic",
    "Raw p-value",
    "Bonferroni p-value",
    "Significant at 5%",
    "N",
]

existing_cols = [c for c in wanted_cols if c in tbl3.columns]
tbl3 = tbl3[existing_cols]

tbl3 = round_for_thesis(tbl3)

tbl3.to_csv(TABLES / "table3_main_dm_microstructure_vs_har.csv", index=False)
print("Saved: table3_main_dm_microstructure_vs_har.csv")


# ── Table 4 — Conditional Microstructure Gains, Small Main Version ───────────
# Uses conditional_df from Section 14B if it already exists.

if "conditional_df" in globals():
    tbl4 = conditional_df[[
        "ETF",
        "Condition",
        "Group",
        "N",
        "M3_vs_M1_pct"
    ]].copy()

    tbl4.columns = [
        "ETF",
        "Condition",
        "Group",
        "N",
        "RMSE improvement (%)"
    ]

    tbl4 = round_for_thesis(tbl4)

    tbl4.to_csv(
        TABLES / "table4_conditional_microstructure_gains.csv",
        index=False
    )

    print("Saved: table4_conditional_microstructure_gains.csv")

else:
    print(
        "Skipped Table 4: conditional_df not found. "
        "Run Section 14B first, then rerun this table block."
    )


# ── Appendix Table A2 — Three-model Loss Comparison ──────────────────────────
# Full model comparison moved to appendix.

a2_rows = []

for etf in ETFS:
    for h in HORIZONS:
        row = {
            "ETF": etf,
            "Horizon": h,
        }

        for m in ["M1", "M2", "M3"]:
            r = loss_df[
                (loss_df.ETF == etf) &
                (loss_df.horizon == h) &
                (loss_df.model == m)
            ].iloc[0]

            row[f"{m} RMSE"] = r.RMSE
            row[f"{m} MAE"] = r.MAE
            row[f"{m} QLIKE"] = r.QLIKE

        a2_rows.append(row)

app_a2 = pd.DataFrame(a2_rows)
app_a2 = round_for_thesis(app_a2)

app_a2.to_csv(TABLES / "appendix_table_A2_three_model_loss.csv", index=False)
print("Saved: appendix_table_A2_three_model_loss.csv")


# ── Appendix Table A3 — All DM Tests ─────────────────────────────────────────
# Full DM test output moved to appendix.

app_a3 = dm_df.copy()

app_a3 = app_a3.rename(columns={
    "horizon": "Horizon",
    "DM_stat": "DM statistic",
    "dm_stat": "DM statistic",
    "p_value_raw": "Raw p-value",
    "p_raw": "Raw p-value",
    "p_value_bonferroni": "Bonferroni p-value",
    "p_bonf": "Bonferroni p-value",
    "significant_5pct": "Significant at 5%",
})

app_a3 = round_for_thesis(app_a3)

app_a3.to_csv(TABLES / "appendix_table_A3_all_dm_tests.csv", index=False)
print("Saved: appendix_table_A3_all_dm_tests.csv")


# ── Notebook Preview ─────────────────────────────────────────────────────────

print("\nPreview — Table 2: Main forecast accuracy, microstructure vs HAR")
print(tbl2.to_string(index=False))

print("\nPreview — Table 3: Main DM tests, microstructure vs HAR")
print(tbl3.to_string(index=False))

if "tbl4" in globals():
    print("\nPreview — Table 4: Conditional microstructure gains")
    print(tbl4.to_string(index=False))

Saved: table1_summary_stats_clean.csv
Saved: table2_main_microstructure_vs_har.csv
Saved: appendix_table_A1_sentiment_vs_har.csv
Saved: table3_main_dm_microstructure_vs_har.csv
Saved: table4_conditional_microstructure_gains.csv
Saved: appendix_table_A2_three_model_loss.csv
Saved: appendix_table_A3_all_dm_tests.csv

Preview — Table 2: Main forecast accuracy, microstructure vs HAR
 ETF  Horizon  HAR RMSE  Microstructure RMSE  RMSE improvement (%)  HAR MAE  Microstructure MAE  MAE improvement (%)   N
 SMH        1    0.0075               0.0068                   8.4   0.0048              0.0046                  5.3 418
 SMH        2    0.0066               0.0061                   8.9   0.0042              0.0040                  5.5 417
 SMH        3    0.0064               0.0059                   8.1   0.0040              0.0037                  7.9 416
 SMH        4    0.0062               0.0059                   6.0   0.0039              0.0036                  6.4 415
 SMH        5

---
## Section 16 — Final Figure Generation

In [18]:
# ── Figure 1 — Data Pipeline Schematic ────────────────────────────────────────


def add_box(ax, x, y, w, h, title, body="", color="#e8e8e8", edge="#333333",
            title_size=8.6, body_size=8.0, bold=True):
    box = mpatches.FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.15",
        facecolor=color,
        edgecolor=edge,
        linewidth=1.2,
    )
    ax.add_patch(box)

    if body:
        ax.text(
            x + w / 2,
            y + h * 0.62,
            title,
            ha="center",
            va="center",
            fontsize=title_size,
            fontweight="bold" if bold else "normal",
        )
        ax.text(
            x + w / 2,
            y + h * 0.34,
            body,
            ha="center",
            va="center",
            fontsize=body_size,
            fontweight="normal",
            linespacing=1.25,
        )
    else:
        ax.text(
            x + w / 2,
            y + h / 2,
            title,
            ha="center",
            va="center",
            fontsize=title_size,
            fontweight="bold" if bold else "normal",
            linespacing=1.25,
        )

def add_arrow(ax, x1, y1, x2, y2):
    ax.annotate(
        "",
        xy=(x2, y2),
        xytext=(x1, y1),
        arrowprops=dict(arrowstyle="->", color="#555555", lw=1.3),
    )

fig, ax = plt.subplots(figsize=(10, 7))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis("off")

# ── Price-data branch ─────────────────────────────────────────────────────────
add_box(
    ax, 0.3, 8.55, 3.5, 0.95,
    "Yahoo Finance",
    "Daily OHLCV data (SMH and SOXX)",
    color="#dbe9f6",
    title_size=9.2,
    body_size=8.4,
)

add_box(
    ax, 0.55, 7.10, 3.0, 0.90,
    "Volatility estimate",
    r"$GK_t$",
    title_size=8.8,
    body_size=8.5,
)

add_box(
    ax, 0.55, 5.65, 3.0, 0.95,
    "HAR lags",
    "Daily, weekly, monthly volatility",
    title_size=8.8,
    body_size=8.0,
)

add_box(
    ax, 0.55, 4.10, 3.0, 1.00,
    "Microstructure indicators",
    "Intraday range and relative volume",
    title_size=8.6,
    body_size=8.0,
)

add_box(
    ax, 0.55, 2.55, 3.0, 0.95,
    "Forecast targets",
    r"$GK_{t,h}$ for $h = 1, 2, 3, 4, 5$",
    title_size=8.8,
    body_size=8.1,
)

for y1, y2 in [(8.55, 8.00), (7.10, 6.60), (5.65, 5.10), (4.10, 3.50)]:
    add_arrow(ax, 2.05, y1, 2.05, y2)

# ── News-data branch ──────────────────────────────────────────────────────────
add_box(
    ax, 6.2, 8.55, 3.5, 0.95,
    "LSEG / Refinitiv",
    "Financial news headlines",
    color="#fde8d0",
    title_size=9.2,
    body_size=8.4,
)

add_box(
    ax, 6.45, 7.10, 3.0, 0.90,
    "Headline cleaning",
    "Deduplication and Trading-day alignment",
    title_size=8.8,
    body_size=8.0,
)

add_box(
    ax, 6.45, 5.65, 3.0, 0.95,
    "FinBERT scoring",
    "Positive, negative, neutral probabilities",
    title_size=8.8,
    body_size=8.0,
)

add_box(
    ax, 6.45, 4.10, 3.0, 1.00,
    "Daily sentiment",
    r"$S_t$ = mean(positive − negative)",
    title_size=8.8,
    body_size=8.0,
)

for y1, y2 in [(8.55, 8.00), (7.10, 6.60), (5.65, 5.10)]:
    add_arrow(ax, 7.95, y1, 7.95, y2)

# ── Final analytical dataset ──────────────────────────────────────────────────
add_box(
    ax, 2.45, 0.95, 5.2, 1.10,
    "Analytical forecasting dataset",
    "SMH and SOXX, 1,508 trading days (2020–2025)",
    color="#d5e8d4",
    edge="#2e7d32",
    title_size=9.2,
    body_size=8.5,
)

# Merge arrows into final dataset
add_arrow(ax, 2.05, 2.55, 3.55, 2.05)
add_arrow(ax, 7.95, 4.10, 6.55, 2.05)

fig.suptitle(
    "Data Pipeline and Analytical Sample Construction",
    fontsize=13,
    fontweight="bold",
    y=0.94,
)

fig.tight_layout(rect=[0, 0.02, 1, 0.96])
fig.savefig(FIGURES / "fig1.1_data_pipeline.png", dpi=150, bbox_inches="tight")
fig.savefig(FIGURES / "fig1.1_data_pipeline.pdf", bbox_inches="tight")
plt.close(fig)

print("Saved fig1.1_data_pipeline.png/.pdf")

Saved fig1.1_data_pipeline.png/.pdf


In [19]:
# ── Figure 1.2 — GK Volatility Time Series ──────────────────────────────────────
events = [
    (pd.Timestamp("2020-03-16"), "COVID\nCrash",       "bottom"),
    (pd.Timestamp("2022-10-14"), "2022\nTech Selloff",       "top"),
    (pd.Timestamp("2025-01-27"), "DeepSeek\nAI Shock",    "top"),
    (pd.Timestamp("2025-04-04"), "Liberation\nDay",    "bottom"),
]

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for ax, etf in zip(axes, ETFS):
    s = gk[etf]
    ax.fill_between(s.index, s.values, alpha=0.22, color=COLORS[etf])
    ax.plot(s.index, s.values, lw=0.8, color=COLORS[etf], label=etf)
    ax.axvline(SPLIT, color="#d73027", lw=1.5, ls="--",
               label="OOS split (May 2024)", zorder=5)
    ax.axvspan(s.index[0], SPLIT, alpha=0.04, color="steelblue")
    ax.axvspan(SPLIT, s.index[-1], alpha=0.04, color="tomato")
    for dt, label, pos in events:
        ax.axvline(dt, color="gray", lw=0.8, ls=":", alpha=0.7)
        y_pos = s.max()*0.95 if pos == "top" else s.max()*0.72
        ax.text(dt, y_pos, label, fontsize=7.5, ha="center",
                va="top", color="dimgray")
    ax.set_ylabel("GK Volatility")
    ax.set_title(f"{etf} — Garman-Klass Daily Volatility (Jan 2020 – Dec 2025)")
    ax.legend(loc="upper left", fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))

axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.text(0.5, -0.01,
         "Shaded areas: in-sample (blue) and out-of-sample (red). "
         "Dashed red line marks OOS split (1 May 2024).",
         ha="center", fontsize=7.5, style="italic")
fig.tight_layout()
for ext in ["png","pdf"]:
    fig.savefig(FIGURES / f"fig1.2_gk_series.{ext}", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved fig1.2_gk_series.png/.pdf")

Saved fig1.2_gk_series.png/.pdf


In [20]:
# ── Figure 2 — OOS Forecast vs Realized (h=1) ─────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
for ax, etf in zip(axes, ETFS):
    real = tgt.loc[oos_idx, f"{etf}_h1"]
    f1   = forecasts[etf]["M1_h1"]
    f2   = forecasts[etf]["M2_h1"]
    f3   = forecasts[etf]["M3_h1"]
    ax.plot(oos_idx, real.values, lw=0.9, color="black", label="Realized GK",
            alpha=0.85, zorder=5)
    ax.plot(oos_idx, f1.values, lw=0.9, color=COLORS["M1"], label="M1 (HAR)")
    ax.plot(oos_idx, f2.values, lw=0.9, color=COLORS["M2"],
            label="M2 (HAR+Sentiment)", alpha=0.8)
    ax.plot(oos_idx, f3.values, lw=0.9, color=COLORS["M3"],
            label="M3 (HAR+Micro)", alpha=0.8, ls="--")
    ax.set_ylabel("GK Volatility")
    ax.set_title(f"{etf} — OOS Forecast vs Realized GK (h = 1)")
    ax.legend(loc="upper right", fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))

axes[-1].xaxis.set_major_locator(mdates.MonthLocator(bymonth=[5,9,1]))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.xticks(rotation=20)
fig.text(0.5, -0.01,
         "OOS period: 1 May 2024 – 31 December 2025.",
         ha="center", fontsize=7.5, style="italic")
fig.tight_layout()
for ext in ["png","pdf"]:
    fig.savefig(FIGURES / f"fig2_oos_forecast.{ext}", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved fig2_oos_forecast.png/.pdf")

Saved fig2_oos_forecast.png/.pdf


In [21]:
# ── Figure 3 — 60-Day Rolling RMSE ────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for ax, etf in zip(axes, ETFS):
    is_mean = float(gk.loc[is_idx, etf].mean())
    real    = tgt.loc[oos_idx, f"{etf}_h1"]
    for model, lbl in [("M1","M1 HAR"),("M2","M2 HAR + FinBERT"),("M3","M3 HAR+Microstructure")]:
        fc  = forecasts[etf][f"{model}_h1"].copy()
        fc[fc <= 0] = is_mean
        roll = ((real - fc)**2).rolling(60, min_periods=20).mean().apply(np.sqrt)
        ax.plot(oos_idx, roll.values, lw=1.2, color=COLORS[model], label=lbl)
    ax.set_ylabel("Rolling RMSE (60-day)")
    ax.set_title(f"{etf} — 60-Day Rolling RMSE: M1 vs M2 vs M3 (h = 1)")
    ax.legend(fontsize=8)

axes[-1].xaxis.set_major_locator(mdates.MonthLocator(bymonth=[5,9,1]))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.xticks(rotation=20)
fig.text(
    0.5, -0.02,
    "Rolling RMSE computed over a 60-day window using h=1 out-of-sample forecasts. Lower values indicate better forecast accuracy.",
    ha="center", fontsize=7.5, style="italic"
)
fig.tight_layout()
for ext in ["png","pdf"]:
    fig.savefig(FIGURES / f"fig3_rolling_rmse.{ext}", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved fig3_rolling_rmse.png/.pdf")

Saved fig3_rolling_rmse.png/.pdf


In [22]:
# ── Figure 4 — M3 Microstructure Coefficient Decay ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=False)
for col_ax, etf in zip(axes, ETFS):
    for var, color, ls, lbl in [
        ("range",  COLORS["M3"], "-",  "Log Range (β_range)"),
        ("relvol", "#d95f02",   "--", "Rel. Volume (β_relvol)"),
    ]:
        coefs, ci_lo, ci_hi = [], [], []
        for h in HORIZONS:
            res = is_results[(etf, h, "M3")]
            if var in res.params.index:
                c, se = res.params[var], res.bse[var]
                coefs.append(c); ci_lo.append(c-1.96*se); ci_hi.append(c+1.96*se)
            else:
                coefs.append(np.nan); ci_lo.append(np.nan); ci_hi.append(np.nan)
        col_ax.plot(HORIZONS, coefs, f"o{ls}", color=color, lw=1.5,
                    markersize=6, label=lbl)
        col_ax.fill_between(HORIZONS, ci_lo, ci_hi, alpha=0.15, color=color)
    col_ax.axhline(0, color="black", lw=0.7, ls="--")
    col_ax.set_xticks(HORIZONS)
    col_ax.set_xlabel("Forecast horizon h")
    col_ax.set_ylabel("Coefficient (M3)")
    col_ax.set_title(f"{etf} — Microstructure Coefficients")
    col_ax.legend(fontsize=8)

fig.text(
    0.5, -0.03,
    "The log high-low range remains positive across horizons, while relative volume is economically close to zero.",
    ha="center", fontsize=7.5, style="italic"
)
fig.suptitle("M3 Microstructure Coefficients Across Forecast Horizons (In-Sample, 95% CI)", y=1.02)
fig.tight_layout()
for ext in ["png","pdf"]:
    fig.savefig(FIGURES / f"fig4_coef_decay.{ext}", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved fig4_coef_decay.png/.pdf")

Saved fig4_coef_decay.png/.pdf


In [23]:
# ── Figure A — Monthly News Coverage ──────────────────────────────────────────
hl_fig = hl_full[hl_full["headline"].notna()].copy()
hl_fig["year"]     = hl_fig["date"].dt.year
hl_fig["month_pd"] = hl_fig["date"].dt.to_period("M")
monthly = hl_fig.groupby("month_pd").size().reset_index(name="count")
monthly["year"]     = monthly["month_pd"].dt.year
monthly["month_dt"] = monthly["month_pd"].dt.to_timestamp()
median_count = monthly["count"].median()

fig, ax = plt.subplots(figsize=(12, 4.5))
for _, row in monthly.iterrows():
    ax.bar(row["month_dt"], row["count"], width=25,
           color=YEAR_COLORS.get(row["year"], "gray"), alpha=0.85, linewidth=0)
ax.axhline(median_count, color="black", ls="--", lw=1.0,
           label=f"Median = {median_count:.0f} headlines/month")
patches = [mpatches.Patch(color=YEAR_COLORS[y], label=str(y))
           for y in sorted(YEAR_COLORS)]
patches.append(Line2D([0],[0], color="black", ls="--", lw=1.0,
                       label=f"Median ({median_count:.0f})"))
ax.legend(handles=patches, loc="upper left", framealpha=0.9, ncol=4)
ax.set_xlabel("Month")
ax.set_ylabel("Number of Headlines")
ax.set_title("Monthly Semiconductor News Coverage (LSEG, 2020-2025)")
ax.set_xlim(monthly["month_dt"].min()-pd.Timedelta(days=15),
            monthly["month_dt"].max()+pd.Timedelta(days=15))
fig.tight_layout()
fig.savefig(FIGURES / "figA_news_coverage.png", bbox_inches="tight")
plt.close(fig)
print("Saved figA_news_coverage.png")

Saved figA_news_coverage.png


In [24]:
# ── Figure B — Annual News Coverage ───────────────────────────────────────────
annual = hl_fig.groupby("year").size().reset_index(name="count")
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(annual["year"].astype(str), annual["count"],
              color=[YEAR_COLORS.get(y, "gray") for y in annual["year"]],
              alpha=0.85, linewidth=0, width=0.6)
for bar, cnt in zip(bars, annual["count"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f"{cnt:,}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Number of Headlines")
ax.set_title("Annual Semiconductor News Coverage (LSEG, 2020–2025)")
ax.set_ylim(0, annual["count"].max() * 1.15)
fig.tight_layout()
fig.savefig(FIGURES / "figB_news_by_year.png", bbox_inches="tight")
plt.close(fig)
print("Saved figB_news_by_year.png")

Saved figB_news_by_year.png


In [25]:
# ── Figure C — Daily Sentiment Time Series ─────────────────────────────────────
sent_plot = cal.copy()
sent_plot = sent_plot[(sent_plot["date"] >= "2020-01-01") &
                       (sent_plot["date"] <= "2025-12-31")].sort_values("date")
s_vals = sent_plot["S_t"].values
d_vals = sent_plot["date"].values
roll30 = pd.Series(s_vals).rolling(30, min_periods=10, center=True).mean().values

fig, ax = plt.subplots(figsize=(12, 4))
pos_mask = s_vals >= 0
ax.fill_between(d_vals, s_vals, 0, where=pos_mask,  alpha=0.20, color="#2ca02c")
ax.fill_between(d_vals, s_vals, 0, where=~pos_mask, alpha=0.20, color="#d62728")
ax.plot(d_vals, s_vals,  color="gray",  lw=0.5, alpha=0.7, label="Daily $S_t$")
ax.plot(d_vals, roll30,  color="navy",  lw=1.5, label="30-day rolling mean")
ax.axhline(0, color="black", lw=0.8)

for ev_date, ev_label in [
    ("2020-03-16", "COVID shock"),
    ("2022-10-14", "2022 Tech Selloff"),
    ("2025-01-27", "DeepSeek AI shock"),
    ("2025-04-02", "Liberation Day tariffs"),
]:
    ts = pd.Timestamp(ev_date)
    ax.axvline(ts, color="black", lw=1.0, ls="--", alpha=0.7)
    ax.text(ts, 0.97, ev_label, transform=ax.get_xaxis_transform(),
            rotation=90, va="top", ha="right", fontsize=7, alpha=0.8)

ax.set_xlabel("Date")
ax.set_ylabel("Net Sentiment Score $S_t$")
ax.set_title("Daily FinBERT Net Sentiment Score $S_t$ (2020-2025)")
ax.legend(loc="lower right")
ax.set_xlim(pd.Timestamp("2020-01-01"), pd.Timestamp("2025-12-31"))
fig.tight_layout()
fig.savefig(FIGURES / "figC_sentiment_series.png", bbox_inches="tight")
plt.close(fig)
print("Saved figC_sentiment_series.png")

Saved figC_sentiment_series.png


In [26]:
# ── Figure D — Sentiment Distribution ─────────────────────────────────────────
s_news = cal.loc[cal["n_headlines"] > 0, "S_t"].values
s_mean = s_news.mean()

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(s_news, bins=50, color="#4E79A7", alpha=0.65, density=True,
        label="Daily $S_t$ (news days)")
kde_x = np.linspace(s_news.min()-0.05, s_news.max()+0.05, 400)
kde   = gaussian_kde(s_news, bw_method="scott")
ax.plot(kde_x, kde(kde_x), color="navy", lw=2.0, label="KDE")
ax.axvline(0,      color="black",   lw=1.2, ls="--", label="Zero")
ax.axvline(s_mean, color="#d62728", lw=1.5, ls="-.",
           label=f"Mean = {s_mean:.3f}")
ax.set_xlabel("Net Sentiment Score $S_t$")
ax.set_ylabel("Density")
ax.set_title("Distribution of Daily FinBERT Net Sentiment (News Days)")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "figD_sentiment_distribution.png", bbox_inches="tight")
plt.close(fig)
print("Saved figD_sentiment_distribution.png")

Saved figD_sentiment_distribution.png


In [27]:
# ── Figure E — Sentiment vs Next-Day GK Volatility ────────────────────────────
merged = cal[["date","S_t"]].merge(
    gk_df[["SMH"]].reset_index().rename(columns={"Date":"date","SMH":"gk_smh"}),
    on="date", how="inner")
merged["next_gk"] = merged["gk_smh"].shift(-1)
merged = merged.dropna(subset=["next_gk","S_t"])
merged = merged[merged["S_t"] != 0]
lo, hi = merged["next_gk"].quantile(0.01), merged["next_gk"].quantile(0.99)
merged = merged[(merged["next_gk"] >= lo) & (merged["next_gk"] <= hi)]

x = merged["S_t"].values
y = merged["next_gk"].values
slope, intercept, r, p, se = sp_stats.linregress(x, y)
x_line = np.linspace(x.min(), x.max(), 200)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(x, y, alpha=0.25, s=10, color="#4E79A7", label="Observations")
p_label = "p<0.001" if p < 0.001 else f"p={p:.3f}"

ax.plot(x_line, intercept + slope*x_line, color="#d62728", lw=2.0,
    label=f"OLS fit (slope={slope:.4f}, r={r:.3f}, {p_label})")

ax.axvline(0, color="black", lw=0.8, ls="--", alpha=0.5)
ax.set_xlabel("Net Sentiment Score $S_t$")
ax.set_ylabel("Next-day GK Volatility (SMH)")
ax.set_title("Net Sentiment vs Next-Day GK Volatility (SMH)")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "figE_sentiment_vs_vol.png", bbox_inches="tight")
plt.close(fig)
print("Saved figE_sentiment_vs_vol.png")

Saved figE_sentiment_vs_vol.png


In [28]:
# ── Figure F — RMSE Comparison M1/M2/M3 (SMH and SOXX) ────────────────────────
x = np.arange(len(HORIZONS))
w = 0.25

pal = {"M1": "#4E79A7", "M2": "#F28E2B", "M3": "#E15759",}
labels = {"M1": "M1 (HAR)", "M2": "M2 (HAR + FinBERT)", "M3": "M3 (HAR + Microstructure)",}
fig, axes = plt.subplots(2, 1, figsize=(8.5, 7), sharex=True)

for ax, etf in zip(axes, ETFS):
    etf_loss = loss_df[loss_df["ETF"] == etf].copy()

    for i, model in enumerate(["M1", "M2", "M3"]):
        rmse_vals = [
            etf_loss.loc[
                (etf_loss["horizon"] == h) & (etf_loss["model"] == model),
                "RMSE"
            ].values[0]
            for h in HORIZONS]
        ax.bar(x + (i - 1) * w, rmse_vals, width=w, label=labels[model], color=pal[model], alpha=0.85, linewidth=0,)

    ax.set_ylabel("RMSE")
    ax.set_title(f"{etf} — Out-of-Sample RMSE by Forecast Horizon")
    ax.set_ylim(0, etf_loss["RMSE"].max() * 1.12)
    ax.grid(axis="y", alpha=0.3, linestyle="--")

axes[-1].set_xticks(x)
axes[-1].set_xticklabels([f"h={h}" for h in HORIZONS])
axes[-1].set_xlabel("Forecast Horizon")

handles, legend_labels = axes[0].get_legend_handles_labels()
fig.legend(handles, legend_labels,title="Model", loc="upper center", ncol=3, frameon=True, bbox_to_anchor=(0.5, 1.02),)

fig.suptitle("Out-of-Sample RMSE: M1 vs M2 vs M3 (May 2024 – Dec 2025)", y=1.06,)

fig.text(0.5, -0.01, "Lower RMSE indicates better forecast accuracy. M2 adds FinBERT sentiment; M3 adds microstructure variables.", 
ha="center", fontsize=7.5, style="italic",)
fig.tight_layout()
fig.savefig(FIGURES / "figF_rmse_comparison.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved figF_rmse_comparison.png")

Saved figF_rmse_comparison.png


---
## Section 17 — Verification and Audit

In [38]:
import math

PASS = "[PASS]"
FAIL = "[FAIL]"
checks = []

def chk(label, condition, detail=""):
    status = PASS if condition else FAIL
    msg = f"{status} {label}"
    if detail:
        msg += f"  ({detail})"
    checks.append((status, label))
    print(msg)

# ── File existence checks ──────────────────────────────────────────────────────
required_files = [
    DATA / "LSEG_headlines_clean.csv",
    DATA / "ohlcv_SMH.csv",
    DATA / "ohlcv_SOXX.csv",
    DATA / "gk_volatility.csv",
    DATA / "forecast_targets.csv",
    DATA / "microstructure.csv",
    DATA / "headlines_scored.csv",
    DATA / "daily_sentiment.csv",

    RESULTS / "forecasts_SMH.csv",
    RESULTS / "forecasts_SOXX.csv",
    RESULTS / "loss_metrics.csv",
    RESULTS / "dm_tests.csv",
    RESULTS / "coefficients.csv",
    RESULTS / "conditional_microstructure_gains.csv",

    TABLES / "table1_summary_stats_clean.csv",
    TABLES / "table2_main_microstructure_vs_har.csv",
    TABLES / "table3_main_dm_microstructure_vs_har.csv",
    TABLES / "table4_conditional_microstructure_gains.csv",
    TABLES / "appendix_table_A1_sentiment_vs_har.csv",
    TABLES / "appendix_table_A2_three_model_loss.csv",
    TABLES / "appendix_table_A3_all_dm_tests.csv",

    FIGURES / "fig1.1_data_pipeline.png",
    FIGURES / "fig1.2_gk_series.png",
    FIGURES / "fig2_oos_forecast.png",
    FIGURES / "fig3_rolling_rmse.png",
    FIGURES / "fig4_coef_decay.png",
    FIGURES / "figG_conditional_microstructure_gains.png",
    FIGURES / "figA_news_coverage.png",
    FIGURES / "figB_news_by_year.png",
    FIGURES / "figC_sentiment_series.png",
    FIGURES / "figD_sentiment_distribution.png",
    FIGURES / "figE_sentiment_vs_vol.png",
    FIGURES / "figF_rmse_comparison.png",
]
for fp in required_files:
    chk(f"File exists: {fp.name}", fp.exists())

# ── Model definition consistency ──────────────────────────────────────────────
m1_vars = set(coef_df[coef_df["model"] == "M1"]["variable"].unique())
m2_vars = set(coef_df[coef_df["model"] == "M2"]["variable"].unique())
m3_vars = set(coef_df[coef_df["model"] == "M3"]["variable"].unique())

chk(
    "M1 = HAR only",
    {"const", "gk_d", "gk_w", "gk_m"}.issubset(m1_vars)
    and "sentiment" not in m1_vars
    and "range" not in m1_vars
    and "relvol" not in m1_vars,
    f"vars={sorted(m1_vars)}")

chk(
    "M2 includes sentiment",
    {"const", "gk_d", "gk_w", "gk_m", "sentiment"}.issubset(m2_vars)
    and "range" not in m2_vars
    and "relvol" not in m2_vars,
    f"vars={sorted(m2_vars)}")

chk(
    "M3 includes range and relvol",
    {"const", "gk_d", "gk_w", "gk_m", "range", "relvol"}.issubset(m3_vars)
    and "sentiment" not in m3_vars,
    f"vars={sorted(m3_vars)}")

# ── No forbidden content ───────────────────────────────────────────────────────
for forbidden in ["gdelt", "M4", "M5", "regime", "LLM"]:
    in_loss = forbidden.lower() in str(loss_df.to_dict()).lower()
    in_dm   = forbidden.lower() in str(dm_df.to_dict()).lower()
    chk(f"No '{forbidden}' in outputs", not in_loss and not in_dm)

# ── Key thesis numerical values ────────────────────────────────────────────────
TOL = 1e-4   # tolerance for comparison to pre-run thesis values
key_vals = [
    ("SMH", 1, "M1", "RMSE", 0.007474),
    ("SMH", 1, "M2", "RMSE", 0.007426),
    ("SMH", 1, "M3", "RMSE", 0.006845),
    ("SOXX",1, "M1", "RMSE", 0.007691),
    ("SOXX",1, "M2", "RMSE", 0.007652),
    ("SOXX",1, "M3", "RMSE", 0.007051),
]
for etf, h, model, metric, expected in key_vals:
    row = loss_df[(loss_df.ETF==etf)&(loss_df.horizon==h)&(loss_df.model==model)]
    if len(row) == 0:
        chk(f"{etf} h={h} {model} {metric}={expected}", False, "row missing")
        continue
    actual = float(row.iloc[0][metric])
    close  = abs(actual - expected) < TOL
    chk(f"{etf} h={h} {model} {metric}={expected:.6f}", close,
        f"actual={actual:.6f}, diff={abs(actual-expected):.2e}")

# ── DM test consistency ────────────────────────────────────────────────────────
m3_m1 = dm_df[dm_df["comparison"]=="M3_vs_M1"]
chk("DM M3 vs M1 has 10 rows (2 ETF × 5 horizon)", len(m3_m1) == 10)

# ── Sentiment sourced from FinBERT ─────────────────────────────────────────────
scored_exists = (DATA / "headlines_scored.csv").exists()
if scored_exists:
    sc = pd.read_csv(DATA / "headlines_scored.csv")
    chk("headlines_scored.csv has net_sentiment column",
        "net_sentiment" in sc.columns)
    chk("Net sentiment in [-1, 1]",
        sc["net_sentiment"].between(-1, 1).all())

# ── Summary ────────────────────────────────────────────────────────────────────
n_pass = sum(1 for s, _ in checks if s == PASS)
n_fail = sum(1 for s, _ in checks if s == FAIL)
print(f"\n{'='*50}")
print(f"Verification: {n_pass} PASS  |  {n_fail} FAIL  |  {len(checks)} total")
if n_fail == 0:
    print("\nReplication notebook completed successfully.")
else:
    failed = [label for s, label in checks if s == FAIL]
    print(f"\nFailed checks: {failed}")

[PASS] File exists: LSEG_headlines_clean.csv
[PASS] File exists: ohlcv_SMH.csv
[PASS] File exists: ohlcv_SOXX.csv
[PASS] File exists: gk_volatility.csv
[PASS] File exists: forecast_targets.csv
[PASS] File exists: microstructure.csv
[PASS] File exists: headlines_scored.csv
[PASS] File exists: daily_sentiment.csv
[PASS] File exists: forecasts_SMH.csv
[PASS] File exists: forecasts_SOXX.csv
[PASS] File exists: loss_metrics.csv
[PASS] File exists: dm_tests.csv
[PASS] File exists: coefficients.csv
[PASS] File exists: conditional_microstructure_gains.csv
[PASS] File exists: table1_summary_stats_clean.csv
[PASS] File exists: table2_main_microstructure_vs_har.csv
[PASS] File exists: table3_main_dm_microstructure_vs_har.csv
[PASS] File exists: table4_conditional_microstructure_gains.csv
[PASS] File exists: appendix_table_A1_sentiment_vs_har.csv
[PASS] File exists: appendix_table_A2_three_model_loss.csv
[PASS] File exists: appendix_table_A3_all_dm_tests.csv
[PASS] File exists: fig1.1_data_pipelin

---
## Section 18 — Supporting Files Summary

In [30]:
print("="*60)
print("GENERATED FILES SUMMARY")
print("="*60)

for label, directory in [("DATA", DATA), ("RESULTS", RESULTS),
                          ("TABLES", TABLES), ("FIGURES", FIGURES)]:
    files = sorted(directory.glob("*"))
    files = [f for f in files if f.is_file()]
    print(f"\n{label}/  ({len(files)} files)")
    for f in files:
        size_kb = f.stat().st_size // 1024
        print(f"  {f.name:<45}  {size_kb:>5} KB")

print("\n" + "="*60)
print("Runtime-sensitive sections:")
print("  Section 8  — FinBERT scoring (~23k headlines)")
print("  Section 12 — Expanding-window OLS (3 models × 2 ETF × 5 h)")
print("\nAssumptions:")
print("  - yfinance data matches 2020-01-01 to 2025-12-31 sample")
print("  - ProsusAI/finbert model downloaded from HuggingFace")
print("  - Numerical results may differ by <1e-4 due to OHLCV rounding")

GENERATED FILES SUMMARY

DATA/  (16 files)
  ._LSEG_headlines_clean.csv                         4 KB
  ._daily_sentiment.csv                              4 KB
  ._forecast_targets.csv                             4 KB
  ._gk_volatility.csv                                4 KB
  ._headlines_scored.csv                             4 KB
  ._microstructure.csv                               4 KB
  ._ohlcv_SMH.csv                                    4 KB
  ._ohlcv_SOXX.csv                                   4 KB
  LSEG_headlines_clean.csv                        2470 KB
  daily_sentiment.csv                              105 KB
  forecast_targets.csv                             321 KB
  gk_volatility.csv                                 77 KB
  headlines_scored.csv                            4279 KB
  microstructure.csv                               131 KB
  ohlcv_SMH.csv                                    137 KB
  ohlcv_SOXX.csv                                   137 KB

RESULTS/  (12 files)
  ._coe